# Downstream Classification Analysis

Downstream classification experiments based on the concept score vector CSVs produced by `[3]concept_score_vector_generation.ipynb`.

1. **Logistic Regression**: train and evaluate separately for 30/50/70/90 concept granularities
2. **Multi-classifier comparison (70-concept)**: XGBoost / SVM / MLP

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATA_DIR = Path('./data')

LABEL_NAMES = [
    'basophil', 'eosinophil', 'erythroblast', 'immature granulocytes',
    'lymphocyte', 'monocyte', 'neutrophil', 'platelet',
]

CONFIGS = [
    {'n': 30, 'dir': 'concept_30', 'prefix': 'concept30'},
    {'n': 50, 'dir': 'concept_50', 'prefix': 'concept50'},
    {'n': 70, 'dir': 'concept_70', 'prefix': 'concept70'},
    {'n': 90, 'dir': 'concept_90', 'prefix': 'concept90'},
]


def load_split_data(cfg):
    """Load merged train+val set and test set. Returns (X_train, y_train, X_test, y_test, feature_cols)."""
    d = DATA_DIR / cfg['dir']
    train_df = pd.read_csv(d / f"{cfg['prefix']}_logits_train.csv")
    val_df   = pd.read_csv(d / f"{cfg['prefix']}_logits_val.csv")
    test_df  = pd.read_csv(d / f"{cfg['prefix']}_logits_test.csv")

    trainval_df = pd.concat([train_df, val_df], ignore_index=True)
    feature_cols = [c for c in trainval_df.columns if c not in ['id', 'label', 'split']]

    scaler = StandardScaler()
    trainval_df[feature_cols] = scaler.fit_transform(trainval_df[feature_cols])
    test_df[feature_cols]     = scaler.transform(test_df[feature_cols])

    X_train = trainval_df[feature_cols].to_numpy(dtype=np.float32)
    y_train = trainval_df['label'].to_numpy(dtype=np.int64)
    X_test  = test_df[feature_cols].to_numpy(dtype=np.float32)
    y_test  = test_df['label'].to_numpy(dtype=np.int64)
    return X_train, y_train, X_test, y_test, feature_cols

# 1 Logistic Regression (30 / 50 / 70 / 90)

In [ ]:
lr_results = {}

for cfg in CONFIGS:
    n = cfg['n']
    print(f'\n{"="*60}')
    print(f'Logistic Regression — {n} concepts')
    print('='*60)

    X_train, y_train, X_test, y_test, feature_cols = load_split_data(cfg)

    clf = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    lr_results[n] = acc
    print(f'Test Accuracy: {acc:.4f}')
    print(classification_report(y_test, y_pred, target_names=LABEL_NAMES))

    # Top-8 positive/negative contributing concepts per class
    coef = clf.coef_
    TOPK = 8
    for cls_idx in range(coef.shape[0]):
        w = coef[cls_idx]
        top_pos = np.argsort(w)[-TOPK:][::-1]
        top_neg = np.argsort(w)[:TOPK]
        print(f'Class {cls_idx} ({LABEL_NAMES[cls_idx]}) top + :')
        for i in top_pos:
            print(f'  + {feature_cols[i]}: {w[i]:.4f}')
        print(f'Class {cls_idx} ({LABEL_NAMES[cls_idx]}) top - :')
        for i in top_neg:
            print(f'  - {feature_cols[i]}: {w[i]:.4f}')
        print()

print('\n--- Summary ---')
for n, acc in lr_results.items():
    print(f'  Concept {n}: {acc:.4f}')

# 2 Multi-Classifier Comparison (70-concept)

Classify using XGBoost / SVM / MLP on the 70-concept granularity.

In [ ]:
# Load 70-concept data (shared by the following three classifiers)
cfg70 = CONFIGS[2]  # n=70
X_train, y_train, X_test, y_test, feature_cols = load_split_data(cfg70)
print(f'70-concept data loaded: train={X_train.shape}, test={X_test.shape}')

## 2.1 XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(
    eval_metric='mlogloss', random_state=42
)
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)

acc_xgb = accuracy_score(y_test, y_pred_xgb)
print(f'XGBoost Test Accuracy: {acc_xgb:.4f}')
print(classification_report(y_test, y_pred_xgb, target_names=LABEL_NAMES))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_xgb))

## 2.2 SVM

In [ ]:
from sklearn.svm import SVC

svm_clf = SVC(kernel='linear', random_state=42)
svm_clf.fit(X_train, y_train)
y_pred_svm = svm_clf.predict(X_test)

acc_svm = accuracy_score(y_test, y_pred_svm)
print(f'SVM Test Accuracy: {acc_svm:.4f}')
print(classification_report(y_test, y_pred_svm, target_names=LABEL_NAMES))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_svm))

## 2.3 MLP

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
mlp_clf.fit(X_train, y_train)
y_pred_mlp = mlp_clf.predict(X_test)

acc_mlp = accuracy_score(y_test, y_pred_mlp)
print(f'MLP Test Accuracy: {acc_mlp:.4f}')
print(classification_report(y_test, y_pred_mlp, target_names=LABEL_NAMES))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_mlp))

## 2.4 Summary Comparison

In [ ]:
summary = pd.DataFrame({
    'Classifier': ['Logistic Regression', 'XGBoost', 'SVM (RBF)', 'MLP'],
    'Accuracy (70-concept)': [
        lr_results.get(70, float('nan')),
        acc_xgb,
        acc_svm,
        acc_mlp,
    ],
})
print(summary.to_string(index=False))